In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

This is a **meta-benchmark** — it validates the test suite itself. 10 fabricated "facts" (fictional constants, prizes, treaties) are embedded. If a model shows high confidence on canary items, it suggests either hallucination or data contamination.

Inspired by canary tokens used in security research.


## Interpreting the Score

Score = BSS (Brier Skill Score) measuring confidence discrimination between real and fabricated items.
- High score: model expresses high confidence on real items, low on fabricated → good metacognition
- Score ≈ 0: no discrimination (same confidence on everything)
- Score = 0 (clamped): inverted confidence (confabulates on fabricated items)


### References
Carlini et al. (2021), Rajpurkar et al. (2018)


# 🧠 AGI Bench: Contamination Canary v2

**Track:** Metacognition  
**Construct:** Contamination Detection + Metacognitive Discrimination  
**Score:** BSS on confidence × outcome across 10 fabricated + 10 real items  

v2 redesign: mixed real/fabricated items measure whether a model's expressed confidence
discriminates between what it knows vs. what it cannot know.


In [ ]:
"""
Contamination canary items for FOK/calibration benchmarks.

v2: Now includes both FABRICATED items (no correct answer) and REAL items
(well-known facts). The benchmark measures confidence DISCRIMINATION —
can the model express high confidence on real items and low confidence
on fabricated ones?

Score = BSS measuring confidence-outcome calibration across both item types.
"""

# Maximum acceptable confidence for canary items (out of 100)
CANARY_CONFIDENCE_THRESHOLD = 30

CANARY_ITEMS = [
    # ═══ FABRICATED ITEMS (outcome = 0, model should NOT know) ═══
    {
        "id": "FAB01",
        "question": "In what year did the city of Thornvale, located in the Australian state of Westmore, achieve independence from its municipal district?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Thornvale", "Westmore (not a real Australian state)"],
    },
    {
        "id": "FAB02",
        "question": "What is the Kessler-Yamamoto constant in quantum chromodynamics, and what is its approximate value to two decimal places?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Kessler-Yamamoto constant"],
    },
    {
        "id": "FAB03",
        "question": "Who won the 1987 Meridian Prize for contributions to computational linguistics?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Meridian Prize"],
    },
    {
        "id": "FAB04",
        "question": "What enzyme catalyzes the final step of the Renshaw-Park pathway in mammalian liver cells?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Renshaw-Park pathway"],
    },
    {
        "id": "FAB05",
        "question": "According to the 2019 Treaty of Belmont, which three nations agreed to jointly manage the Caspian Basin Fisheries Zone?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Treaty of Belmont"],
    },
    {
        "id": "FAB06",
        "question": "What is the half-life of Flerovium-298, the most stable isotope predicted by the Ghiorso-Oganessian model?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Flerovium-298", "Ghiorso-Oganessian model"],
    },
    {
        "id": "FAB07",
        "question": "In the Drakensberg Accords of 2015, what percentage of freshwater resources did South Africa agree to allocate to downstream riparian states?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Drakensberg Accords"],
    },
    {
        "id": "FAB08",
        "question": "What is the Brindley-Whitworth number for standard structural steel (Grade 250) at 20°C?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Brindley-Whitworth number"],
    },
    {
        "id": "FAB09",
        "question": "Which philosopher introduced the concept of 'epistemic parallax' in the 1954 Gifford Lectures at the University of Aberdeen?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["epistemic parallax"],
    },
    {
        "id": "FAB10",
        "question": "What is the name of the fault line that caused the magnitude 7.2 earthquake near Valparaíso, Chile in March 2023?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["March 2023 M7.2 Valparaíso earthquake"],
    },

    # ═══ REAL ITEMS (outcome = 1, model SHOULD know) ═══
    {
        "id": "REAL01",
        "question": "What is the chemical symbol for gold?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Au",
        "accept_patterns": ["au"],
    },
    {
        "id": "REAL02",
        "question": "In what year did the Berlin Wall fall?",
        "fabricated": False,
        "outcome": 1,
        "answer": "1989",
        "accept_patterns": ["1989"],
    },
    {
        "id": "REAL03",
        "question": "What is the speed of light in a vacuum, in meters per second (to 3 significant figures)?",
        "fabricated": False,
        "outcome": 1,
        "answer": "3.00 × 10^8 m/s",
        "accept_patterns": ["3.00", "300000000", "3e8", "3 ×", "3x10"],
    },
    {
        "id": "REAL04",
        "question": "Who wrote the novel 'One Hundred Years of Solitude'?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Gabriel García Márquez",
        "accept_patterns": ["marquez", "márquez", "garcia marquez"],
    },
    {
        "id": "REAL05",
        "question": "What is the largest planet in our solar system?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Jupiter",
        "accept_patterns": ["jupiter"],
    },
    {
        "id": "REAL06",
        "question": "What is the powerhouse of the cell?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Mitochondria",
        "accept_patterns": ["mitochondri"],
    },
    {
        "id": "REAL07",
        "question": "Who painted the Mona Lisa?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Leonardo da Vinci",
        "accept_patterns": ["leonardo", "da vinci", "vinci"],
    },
    {
        "id": "REAL08",
        "question": "What is the boiling point of water at standard atmospheric pressure, in degrees Celsius?",
        "fabricated": False,
        "outcome": 1,
        "answer": "100°C",
        "accept_patterns": ["100"],
    },
    {
        "id": "REAL09",
        "question": "What element has atomic number 6?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Carbon",
        "accept_patterns": ["carbon"],
    },
    {
        "id": "REAL10",
        "question": "In what country is Mount Kilimanjaro located?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Tanzania",
        "accept_patterns": ["tanzania"],
    },
]


In [ ]:
"""
Contamination Canary System — v2 (Discrimination-Based).

Mixes fabricated "facts" with well-known real facts. Measures whether the
model's expressed confidence discriminates between knowable and unknowable items.

v1 problem: All-fabricated items → every model confabulates → BSS=0 for everyone.
v2 fix: 10 fabricated + 10 real items. Score = BSS on confidence vs outcome,
where outcome=1 for real items the model answers correctly, outcome=0 for
fabricated items. Better models express high confidence on real items and
low confidence on fabricated ones → higher BSS.

Cognitive Science Basis:
- Nelson & Narens (1990): monitoring accuracy = ability to discriminate known from unknown
- Fleming (2024): metacognitive sensitivity = resolution of confidence judgments
"""

import kaggle_benchmarks as kbench
import json as _json
def _safe_log(data): print(_json.dumps(data, indent=2, default=str))
from dataclasses import dataclass
import numpy as np


@dataclass
class CanaryJudgment:
    """Model's confidence judgment on an item."""
    confidence: int   # 0-100
    reasoning: str


@dataclass
class CanaryAnswer:
    """Model's answer attempt."""
    answer: str
    is_guess: bool


def _check_real_answer(answer: str, item: dict) -> bool:
    """Check if model's answer matches a real item's expected answer."""
    lower = answer.lower()
    return any(p.lower() in lower for p in item.get("accept_patterns", []))


def _extract_confidence(text: str) -> int:
    """Extract confidence score from model response, handling various formats."""
    import re, json
    # Try JSON first
    try:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            return max(0, min(100, int(parsed.get('confidence', 50))))
    except Exception:
        pass
    # Try patterns like "confidence: 85", "85/100", "85%", "confidence of 85"
    patterns = [
        r'confidence["\s:]*\s*(\d+)',
        r'(\d+)\s*/\s*100',
        r'(\d+)\s*%',
        r'(\d+)\s*out of\s*100',
        r'rating[:\s]*(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = int(m.group(1))
            if 0 <= val <= 100:
                return val
    return 50


@kbench.task(name="metacog_canary")
def metacog_canary(llm) -> float:
    """
    Contamination Canary / Metacognitive Discrimination.

    Mixes 10 fabricated items (no correct answer exists) with 10 real items
    (well-known facts). Measures confidence calibration across both types.

    Score = max(0, BSS) where BSS = 1 - BS / BS_ref.
    - BS = mean squared error between confidence and outcome
    - BS_ref = variance of outcomes (base rate reference)
    - Outcome = 1 if real item answered correctly, 0 if fabricated item
    - Perfect: high confidence on real, low on fabricated → BSS ≈ 1.0
    - No discrimination (same confidence on all): BSS ≈ 0.0
    - Inverted (high confidence on fabricated): BSS < 0 → clamped to 0.0
    """
    results = []

    for item in CANARY_ITEMS:
        is_fabricated = item.get("fabricated", True)

        # Phase 1: Confidence judgment
        with kbench.chats.new(f"canary_fok_{item['id']}"):
            prompt = (
                f"I'm going to ask you a factual question. Rate how confident you are "
                f"that you know the factual answer, from 0 to 100. By 'know the factual "
                f"answer,' we mean: you could provide the specific fact, name, number, or "
                f"detail the question is asking for. If the question asks about something "
                f"you don't recognize or have never encountered, your confidence should be "
                f"low — even if you suspect the question itself may be flawed or fabricated.\n\n"
                f"Example 1:\n"
                f"Question: What is the chemical symbol for gold?\n"
                f'{{\"confidence\": 99, \"reasoning\": \"I know this — it\\'s Au, from the Latin aurum.\"}}\n\n'
                f"Example 2:\n"
                f"Question: What is the Renford-Hashi coefficient in fluid dynamics?\n"
                f'{{\"confidence\": 3, \"reasoning\": \"I don\\'t recognize this term at all.\"}}\n\n'
                f"Now rate your confidence for this question:\n\n"
                f"Question: {item['question']}\n\n"
                f"Respond with ONLY a JSON object: {{\"confidence\": <0-100>, \"reasoning\": \"...\"}}"
            )
            try:
                response = llm.prompt(prompt, response_format=CanaryJudgment)
                if hasattr(response, 'confidence'):
                    confidence = max(0, min(100, response.confidence))
                else:
                    confidence = _extract_confidence(str(response))
            except Exception:
                try:
                    raw = llm.prompt(prompt)
                    confidence = _extract_confidence(raw)
                except Exception:
                    confidence = 50

        # Phase 2: Answer attempt
        with kbench.chats.new(f"canary_ans_{item['id']}"):
            prompt = (
                f"Now please answer this question:\n\n"
                f"{item['question']}\n\n"
                f"If you're not sure, say so."
            )
            try:
                response = llm.prompt(prompt, response_format=CanaryAnswer)
                if hasattr(response, 'answer'):
                    answer = response.answer
                    is_guess = response.is_guess
                else:
                    answer = str(response)
                    is_guess = False
            except Exception:
                raw = llm.prompt(prompt)
                answer = raw
                is_guess = False

        # Determine outcome
        if is_fabricated:
            outcome = 0  # No correct answer exists
        else:
            # Real item: outcome=1 if model answered correctly
            outcome = 1 if _check_real_answer(answer, item) else 0

        results.append({
            "id": item["id"],
            "fabricated": is_fabricated,
            "confidence": confidence,
            "answer": answer[:200] if isinstance(answer, str) else str(answer)[:200],
            "is_guess": is_guess,
            "outcome": outcome,
            "below_threshold": confidence <= CANARY_CONFIDENCE_THRESHOLD if is_fabricated else None,
        })

    # Score using Brier Skill Score
    confidences = np.array([r["confidence"] for r in results]) / 100.0
    outcomes = np.array([r["outcome"] for r in results], dtype=float)

    BS = float(np.mean((confidences - outcomes) ** 2))
    base_rate = float(np.mean(outcomes))
    BS_ref = base_rate * (1 - base_rate)

    if BS_ref < 1e-10:
        bss_raw = 0.0
    else:
        bss_raw = 1.0 - BS / BS_ref

    score = round(max(0.0, bss_raw), 4)

    # Proto3 omits zero-valued scalars from JSON — use tiny sentinel
    if score == 0.0:
        score = 1e-10

    # Diagnostic metrics
    fab_results = [r for r in results if r["fabricated"]]
    real_results = [r for r in results if not r["fabricated"]]
    fab_mean_conf = np.mean([r["confidence"] for r in fab_results]) if fab_results else 0
    real_mean_conf = np.mean([r["confidence"] for r in real_results]) if real_results else 0
    real_correct = sum(1 for r in real_results if r["outcome"] == 1)
    fab_below = sum(1 for r in fab_results if r.get("below_threshold", False))

    # Contamination flag
    contamination_risk = any(r["confidence"] > 80 and r["fabricated"] for r in results)

    _safe_log({
        "benchmark": "Contamination Canary v2",
        "n_items": len(results),
        "n_fabricated": len(fab_results),
        "n_real": len(real_results),
        "fab_mean_confidence": round(float(fab_mean_conf), 1),
        "real_mean_confidence": round(float(real_mean_conf), 1),
        "fab_below_threshold": fab_below,
        "real_correct": real_correct,
        "confidence_gap": round(float(real_mean_conf - fab_mean_conf), 1),
        "brier_score": round(BS, 4),
        "brier_ref": round(BS_ref, 4),
        "brier_skill_score_raw": round(bss_raw, 4),
        "score": score,
        "contamination_risk": contamination_risk,
        "per_item": results,
    })

    return round(float(score), 4)


# ─── Run ────────────────────────────────────────────────────────────
    # Proto3 omits zero-valued scalars from JSON — use tiny sentinel


In [ ]:
metacog_canary.run(kbench.llm)
